In [0]:
%run ../99_utils/utils_config

UTILS CONFIG LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: 41894d08-8487-4f3a-acbf-a2f23e932909


In [0]:
%run ../99_utils/utils_quality

UTILS CONFIG LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: 360d896d-cabf-4413-9cf7-901ec10837b1


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 Quality — Traceability Checks
# MAGIC
# MAGIC Executes cross-layer traceability validations for the Brazil Legislative Analytics Medallion project.
# MAGIC
# MAGIC ## Purpose
# MAGIC This notebook validates whether pipeline traceability fields are consistently available across Bronze, Silver and Gold layers.
# MAGIC
# MAGIC ## Scope
# MAGIC Traceability checks focus on:
# MAGIC - table existence by layer
# MAGIC - required traceability column availability
# MAGIC - execution identifier availability
# MAGIC - pipeline version availability
# MAGIC - processing timestamp availability
# MAGIC - record hash availability when applicable
# MAGIC - controlled exception handling per entity
# MAGIC
# MAGIC ## Persistence
# MAGIC Validation results are persisted into:
# MAGIC
# MAGIC ```text
# MAGIC audit.aud_log_qualidade_dados
# MAGIC ```
# MAGIC
# MAGIC ## Execution Policy
# MAGIC During early development, some pipeline tables may not exist yet.
# MAGIC In this case, validations are persisted as evidence, but the notebook does not block execution.
# MAGIC
# MAGIC Set `FAIL_ON_ERROR = True` when all layers are active and traceability checks must block the pipeline.
# MAGIC
# MAGIC ## Documentation Standard
# MAGIC - Python functions and variables are written in English.
# MAGIC - Table and field names follow Portuguese mnemonic standards.
# MAGIC - Comments and documentation are written in English.

# COMMAND ----------

# MAGIC %run ../99_utils/utils_config

# COMMAND ----------

# MAGIC %run ../99_utils/utils_quality

# COMMAND ----------

from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# COMMAND ----------

print("=" * 90)
print("BRAZIL LEGISLATIVE ANALYTICS MEDALLION")
print("04 - TRACEABILITY CHECKS")
print("=" * 90)
print(f"Execution Timestamp: {datetime.now()}")
print(f"Catalog: {CATALOG_NAME}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# QUALITY CONFIGURATION
# ============================================================

NOTEBOOK_NAME = "04_traceability_checks"
LAYER_NAME = "quality"

# During early development, some tables may not exist yet.
# Set to True when all layers are active and traceability checks
# must block the pipeline.
FAIL_ON_ERROR = False

DATA_QUALITY_LOG_TABLE = (
    f"{CATALOG_NAME}."
    f"{SCHEMA_AUDIT}."
    f"{AUD_TB_LOG_QUALIDADE_DADOS}"
)

TRACEABILITY_RULES = {
    "bronze": {
        "schema_name": SCHEMA_BRONZE,
        "tables": BRONZE_TABLES,
        "required_columns": [
            "aud_id_execucao",
            "aud_dh_ingestao",
            "aud_tx_endpoint_origem",
            "aud_tx_sistema_origem",
            "aud_tx_versao_pipeline",
            "aud_tx_hash_registro",
        ],
        "critical_columns": [
            "aud_id_execucao",
            "aud_dh_ingestao",
            "aud_tx_endpoint_origem",
            "aud_tx_hash_registro",
        ],
    },
    "silver": {
        "schema_name": SCHEMA_SILVER,
        "tables": SILVER_TABLES,
        "required_columns": [
            "aud_id_execucao",
            "aud_dh_processamento",
            "aud_tx_versao_pipeline",
            "aud_tx_hash_registro",
        ],
        "critical_columns": [
            "aud_id_execucao",
            "aud_dh_processamento",
            "aud_tx_hash_registro",
        ],
    },
    "gold": {
        "schema_name": SCHEMA_GOLD,
        "tables": {
            **GOLD_DIMENSION_TABLES,
            **GOLD_FACT_TABLES,
        },
        "required_columns": [
            "aud_id_execucao",
            "aud_dh_processamento",
            "aud_tx_versao_pipeline",
        ],
        "critical_columns": [
            "aud_id_execucao",
            "aud_dh_processamento",
        ],
    },
}

quality_results = []

# COMMAND ----------

# ============================================================
# QUALITY HELPERS
# ============================================================

def add_quality_result(
    rule_name: str,
    rule_description: str,
    validation_status: str,
    total_records: int,
    invalid_records: int,
    invalid_percentage: float,
    message: str,
    entity_name: str,
    target_table: str,
) -> None:
    """
    Adds a quality validation result to the in-memory result list.
    """

    quality_results.append({
        "nome_regra": rule_name,
        "descricao_regra": rule_description,
        "status_validacao": validation_status,
        "total_registros": int(total_records) if total_records is not None else 0,
        "registros_invalidos": int(invalid_records) if invalid_records is not None else 0,
        "percentual_invalidos": float(invalid_percentage) if invalid_percentage is not None else 0.0,
        "mensagem": message,
        "entity_name": entity_name,
        "target_table": target_table,
    })

# COMMAND ----------

def add_exception_result(
    entity_name: str,
    target_table: str,
    error: Exception,
) -> None:
    """
    Adds a controlled exception result to the quality result list.
    """

    add_quality_result(
        rule_name="traceability_quality_exception",
        rule_description="Captures unexpected errors during traceability validation.",
        validation_status=QUALITY_FAILED,
        total_records=1,
        invalid_records=1,
        invalid_percentage=100.0,
        message=f"Unexpected error during traceability validation: {str(error)}",
        entity_name=entity_name,
        target_table=target_table,
    )

# COMMAND ----------

def table_exists(
    full_table_name: str,
) -> bool:
    """
    Checks whether a fully qualified table exists.
    """

    try:
        return spark.catalog.tableExists(full_table_name)

    except Exception:
        return False

# COMMAND ----------

def get_table_dataframe(
    full_table_name: str,
) -> DataFrame:
    """
    Reads a table into a Spark DataFrame.
    """

    return spark.table(full_table_name)

# COMMAND ----------

def get_full_table_name_by_layer(
    schema_name: str,
    table_name: str,
) -> str:
    """
    Builds a fully qualified table name for any project layer.
    """

    return (
        f"{CATALOG_NAME}."
        f"{schema_name}."
        f"{table_name}"
    )

# COMMAND ----------

def validate_table_exists(
    layer_name: str,
    entity_name: str,
    full_table_name: str,
) -> bool:
    """
    Validates whether a table exists for traceability checks.
    """

    exists = table_exists(full_table_name)

    add_quality_result(
        rule_name="traceability_table_exists",
        rule_description="Validates whether the table exists for traceability checks.",
        validation_status=QUALITY_PASSED if exists else QUALITY_FAILED,
        total_records=1,
        invalid_records=0 if exists else 1,
        invalid_percentage=0.0 if exists else 100.0,
        message=(
            "Table exists."
            if exists
            else "Table does not exist."
        ),
        entity_name=f"{layer_name}.{entity_name}",
        target_table=full_table_name,
    )

    return exists

# COMMAND ----------

def validate_required_traceability_columns(
    dataframe: DataFrame,
    layer_name: str,
    entity_name: str,
    full_table_name: str,
    required_columns: list,
) -> None:
    """
    Validates required traceability columns for a table.
    """

    result = validate_required_columns(
        dataframe=dataframe,
        required_columns=required_columns,
    )

    add_quality_result(
        rule_name="traceability_required_columns",
        rule_description="Validates required traceability columns.",
        validation_status=result["status_validacao"],
        total_records=result["total_registros"],
        invalid_records=result["registros_invalidos"],
        invalid_percentage=result["percentual_invalidos"],
        message=result["mensagem"],
        entity_name=f"{layer_name}.{entity_name}",
        target_table=full_table_name,
    )

# COMMAND ----------

def validate_column_population(
    dataframe: DataFrame,
    layer_name: str,
    entity_name: str,
    full_table_name: str,
    column_name: str,
) -> None:
    """
    Validates whether a traceability column is populated.
    """

    if column_name not in dataframe.columns:

        add_quality_result(
            rule_name=f"traceability_population_{column_name}",
            rule_description="Validates whether a traceability column is populated.",
            validation_status=QUALITY_FAILED,
            total_records=1,
            invalid_records=1,
            invalid_percentage=100.0,
            message=f"Column does not exist: {column_name}",
            entity_name=f"{layer_name}.{entity_name}",
            target_table=full_table_name,
        )

        return

    total_records = dataframe.count()

    invalid_records = (
        dataframe
        .filter(
            F.col(column_name).isNull()
            | (F.trim(F.col(column_name).cast("string")) == "")
        )
        .count()
    )

    invalid_percentage = (
        0.0 if total_records == 0
        else round((invalid_records / total_records) * 100, 2)
    )

    validation_status = (
        QUALITY_PASSED
        if invalid_records == 0
        else QUALITY_WARNING
    )

    add_quality_result(
        rule_name=f"traceability_population_{column_name}",
        rule_description="Validates whether a traceability column is populated.",
        validation_status=validation_status,
        total_records=total_records,
        invalid_records=invalid_records,
        invalid_percentage=invalid_percentage,
        message=f"Records with missing {column_name}: {invalid_records}",
        entity_name=f"{layer_name}.{entity_name}",
        target_table=full_table_name,
    )

# COMMAND ----------

def run_entity_checks(
    layer_name: str,
    schema_name: str,
    entity_name: str,
    table_name: str,
    required_columns: list,
    critical_columns: list,
) -> None:
    """
    Executes traceability checks for a single table.
    """

    full_table_name = get_full_table_name_by_layer(
        schema_name=schema_name,
        table_name=table_name,
    )

    print("=" * 90)
    print(f"Running traceability checks for: {full_table_name}")
    print("=" * 90)

    try:

        if not validate_table_exists(
            layer_name=layer_name,
            entity_name=entity_name,
            full_table_name=full_table_name,
        ):
            return

        dataframe = get_table_dataframe(full_table_name)

        validate_required_traceability_columns(
            dataframe=dataframe,
            layer_name=layer_name,
            entity_name=entity_name,
            full_table_name=full_table_name,
            required_columns=required_columns,
        )

        for column_name in critical_columns:

            validate_column_population(
                dataframe=dataframe,
                layer_name=layer_name,
                entity_name=entity_name,
                full_table_name=full_table_name,
                column_name=column_name,
            )

    except Exception as error:

        add_exception_result(
            entity_name=f"{layer_name}.{entity_name}",
            target_table=full_table_name,
            error=error,
        )

# COMMAND ----------

def build_traceability_quality_log() -> DataFrame:
    """
    Builds the final traceability quality log DataFrame.
    """

    if not quality_results:

        add_quality_result(
            rule_name="traceability_no_results",
            rule_description="Validates whether traceability checks produced results.",
            validation_status=QUALITY_WARNING,
            total_records=0,
            invalid_records=0,
            invalid_percentage=0.0,
            message="No traceability quality results were generated.",
            entity_name="traceability",
            target_table=DATA_QUALITY_LOG_TABLE,
        )

    quality_base_df = spark.createDataFrame(
        quality_results
    )

    return (
        quality_base_df
        .withColumn("qlt_id_log", F.expr("uuid()"))
        .withColumn("aud_id_execucao", F.lit(RUN_ID))
        .withColumn("aud_tx_nome_projeto", F.lit(PROJECT_NAME))
        .withColumn("aud_tx_versao_pipeline", F.lit(PROJECT_VERSION))
        .withColumn("aud_tx_ambiente", F.lit(PROJECT_ENVIRONMENT))
        .withColumn("aud_tx_nome_notebook", F.lit(NOTEBOOK_NAME))
        .withColumn("aud_tx_nome_camada", F.lit(LAYER_NAME))
        .withColumn("aud_tx_nome_entidade", F.col("entity_name"))
        .withColumn("aud_tx_tabela_destino", F.col("target_table"))
        .withColumn("qlt_tx_nome_regra", F.col("nome_regra"))
        .withColumn("qlt_tx_descricao_regra", F.col("descricao_regra"))
        .withColumn("qlt_tx_status_validacao", F.col("status_validacao"))
        .withColumn("qlt_qt_total_registros", F.col("total_registros"))
        .withColumn("qlt_qt_registros_invalidos", F.col("registros_invalidos"))
        .withColumn("qlt_pc_registros_invalidos", F.col("percentual_invalidos"))
        .withColumn("qlt_dh_validacao", F.current_timestamp())
        .withColumn("qlt_tx_mensagem", F.col("mensagem"))
        .select(
            "qlt_id_log",
            "aud_id_execucao",
            "aud_tx_nome_projeto",
            "aud_tx_versao_pipeline",
            "aud_tx_ambiente",
            "aud_tx_nome_notebook",
            "aud_tx_nome_camada",
            "aud_tx_nome_entidade",
            "aud_tx_tabela_destino",
            "qlt_tx_nome_regra",
            "qlt_tx_descricao_regra",
            "qlt_tx_status_validacao",
            "qlt_qt_total_registros",
            "qlt_qt_registros_invalidos",
            "qlt_pc_registros_invalidos",
            "qlt_dh_validacao",
            "qlt_tx_mensagem",
        )
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Execute Traceability Checks

# COMMAND ----------

for layer_name, layer_config in TRACEABILITY_RULES.items():

    for entity_name, table_name in layer_config["tables"].items():

        run_entity_checks(
            layer_name=layer_name,
            schema_name=layer_config["schema_name"],
            entity_name=entity_name,
            table_name=table_name,
            required_columns=layer_config["required_columns"],
            critical_columns=layer_config["critical_columns"],
        )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Persist Quality Results

# COMMAND ----------

quality_log_df = build_traceability_quality_log()

quality_log_df.write.mode(
    "append"
).saveAsTable(DATA_QUALITY_LOG_TABLE)

print(
    f"Traceability quality results persisted into: "
    f"{DATA_QUALITY_LOG_TABLE}"
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Display Quality Results

# COMMAND ----------

display(quality_log_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Quality Summary

# COMMAND ----------

failed_count = (
    quality_log_df
    .filter("qlt_tx_status_validacao = 'FAILED'")
    .count()
)

warning_count = (
    quality_log_df
    .filter("qlt_tx_status_validacao = 'WARNING'")
    .count()
)

passed_count = (
    quality_log_df
    .filter("qlt_tx_status_validacao = 'PASSED'")
    .count()
)

print("=" * 90)
print("TRACEABILITY QUALITY SUMMARY")
print("=" * 90)
print(f"Passed validations: {passed_count}")
print(f"Warning validations: {warning_count}")
print(f"Failed validations: {failed_count}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# QUALITY EXECUTION POLICY
# ============================================================

if failed_count > 0 and FAIL_ON_ERROR:

    raise Exception(
        f"Traceability validation failed with "
        f"{failed_count} failed validation(s)."
    )

if failed_count > 0:

    print(
        f"WARNING: Traceability validation finished with "
        f"{failed_count} failed validation(s). "
        "This is expected if some pipeline tables have not been created yet."
    )

print("TRACEABILITY CHECKS COMPLETED")

BRAZIL LEGISLATIVE ANALYTICS MEDALLION
04 - TRACEABILITY CHECKS
Execution Timestamp: 2026-05-20 03:21:06.746294
Catalog: brazil_legislative_analytics
Running traceability checks for: brazil_legislative_analytics.bronze.br_deputados
Running traceability checks for: brazil_legislative_analytics.bronze.br_frentes
Running traceability checks for: brazil_legislative_analytics.bronze.br_frentes_membros
Running traceability checks for: brazil_legislative_analytics.bronze.br_frentes_detalhes
Running traceability checks for: brazil_legislative_analytics.bronze.br_eventos
Running traceability checks for: brazil_legislative_analytics.bronze.br_votacoes
Running traceability checks for: brazil_legislative_analytics.bronze.br_votos
Running traceability checks for: brazil_legislative_analytics.bronze.br_despesas_ceap
Running traceability checks for: brazil_legislative_analytics.bronze.br_cpis
Running traceability checks for: brazil_legislative_analytics.bronze.br_cpi_eventos
Running traceability chec

qlt_id_log,aud_id_execucao,aud_tx_nome_projeto,aud_tx_versao_pipeline,aud_tx_ambiente,aud_tx_nome_notebook,aud_tx_nome_camada,aud_tx_nome_entidade,aud_tx_tabela_destino,qlt_tx_nome_regra,qlt_tx_descricao_regra,qlt_tx_status_validacao,qlt_qt_total_registros,qlt_qt_registros_invalidos,qlt_pc_registros_invalidos,qlt_dh_validacao,qlt_tx_mensagem
2ea4f056-307f-47e2-80ae-625e23dd276f,360d896d-cabf-4413-9cf7-901ec10837b1,brazil_legislative_analytics,v1.0.0,dev,04_traceability_checks,quality,bronze.deputados,brazil_legislative_analytics.bronze.br_deputados,traceability_table_exists,Validates whether the table exists for traceability checks.,FAILED,1,1,100.0,2026-05-20T03:21:15.083Z,Table does not exist.
d23666e5-1c62-49b5-9124-bf29a305c35e,360d896d-cabf-4413-9cf7-901ec10837b1,brazil_legislative_analytics,v1.0.0,dev,04_traceability_checks,quality,bronze.frentes,brazil_legislative_analytics.bronze.br_frentes,traceability_table_exists,Validates whether the table exists for traceability checks.,FAILED,1,1,100.0,2026-05-20T03:21:15.083Z,Table does not exist.
d8c0f48e-9c3e-4fbc-96f8-8c72195a076d,360d896d-cabf-4413-9cf7-901ec10837b1,brazil_legislative_analytics,v1.0.0,dev,04_traceability_checks,quality,bronze.frentes_membros,brazil_legislative_analytics.bronze.br_frentes_membros,traceability_table_exists,Validates whether the table exists for traceability checks.,FAILED,1,1,100.0,2026-05-20T03:21:15.083Z,Table does not exist.
802dc317-4609-48b8-8910-cae0df7cc028,360d896d-cabf-4413-9cf7-901ec10837b1,brazil_legislative_analytics,v1.0.0,dev,04_traceability_checks,quality,bronze.frentes_detalhes,brazil_legislative_analytics.bronze.br_frentes_detalhes,traceability_table_exists,Validates whether the table exists for traceability checks.,FAILED,1,1,100.0,2026-05-20T03:21:15.083Z,Table does not exist.
567bdfb7-8688-47dc-b34b-e955ad46c171,360d896d-cabf-4413-9cf7-901ec10837b1,brazil_legislative_analytics,v1.0.0,dev,04_traceability_checks,quality,bronze.eventos,brazil_legislative_analytics.bronze.br_eventos,traceability_table_exists,Validates whether the table exists for traceability checks.,FAILED,1,1,100.0,2026-05-20T03:21:15.083Z,Table does not exist.
b4ccf78d-e230-41c6-aa8b-87f345edbef2,360d896d-cabf-4413-9cf7-901ec10837b1,brazil_legislative_analytics,v1.0.0,dev,04_traceability_checks,quality,bronze.votacoes,brazil_legislative_analytics.bronze.br_votacoes,traceability_table_exists,Validates whether the table exists for traceability checks.,FAILED,1,1,100.0,2026-05-20T03:21:15.083Z,Table does not exist.
0f518260-808f-4e05-b838-01fae9060359,360d896d-cabf-4413-9cf7-901ec10837b1,brazil_legislative_analytics,v1.0.0,dev,04_traceability_checks,quality,bronze.votos,brazil_legislative_analytics.bronze.br_votos,traceability_table_exists,Validates whether the table exists for traceability checks.,FAILED,1,1,100.0,2026-05-20T03:21:15.083Z,Table does not exist.
8621a519-385b-48e3-9613-203b1696ed35,360d896d-cabf-4413-9cf7-901ec10837b1,brazil_legislative_analytics,v1.0.0,dev,04_traceability_checks,quality,bronze.despesas_ceap,brazil_legislative_analytics.bronze.br_despesas_ceap,traceability_table_exists,Validates whether the table exists for traceability checks.,FAILED,1,1,100.0,2026-05-20T03:21:15.083Z,Table does not exist.
1ac2fd77-93ef-4db3-9293-4655761a2613,360d896d-cabf-4413-9cf7-901ec10837b1,brazil_legislative_analytics,v1.0.0,dev,04_traceability_checks,quality,bronze.cpis,brazil_legislative_analytics.bronze.br_cpis,traceability_table_exists,Validates whether the table exists for traceability checks.,FAILED,1,1,100.0,2026-05-20T03:21:15.083Z,Table does not exist.
64dca1a1-df2d-48d5-aa39-b4058cca60e0,360d896d-cabf-4413-9cf7-901ec10837b1,brazil_legislative_analytics,v1.0.0,dev,04_traceability_checks,quality,bronze.cpi_eventos,brazil_legislative_analytics.bronze.br_cpi_eventos,traceability_table_exists,Validates whether the table exists for traceability checks.,FAILED,1,1,100.0,2026-05-20T03:21:15.083Z,Table does not exist.


TRACEABILITY QUALITY SUMMARY
Passed validations: 0
Warning validations: 0
Failed validations: 41
TRACEABILITY CHECKS COMPLETED


utils_quality loaded successfully.
